<a href="https://colab.research.google.com/github/yoeda11/Ddd/blob/main/AS_NaivaBayes_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas

In [2]:
import pandas as pd

filepath = "/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/comments_2.csv"
ytb = pd.read_csv(filepath)
# ytb.head()
# ytb.info()

In [3]:
# CASE FOLDING

ytb["hasil_casefolding"] = ytb["comment"].str.lower()
# ytb.head()

In [4]:
# CLEANING PEMBERSIHAN TEXT

import re

def clean_text(text):
  if not isinstance(text, str):
    return ""

  # 1. Case Folding
  # text = text.lower() # sudah ada di atas

  # 2. Hapus URL
  text = re.sub(r"https?://\S+|www\.\S+", " ", text)

  # # 3. Hapus mention & hashtag
  # text = re.sub(r"@\w+|#\w+", " ", text)
  text = re.sub(r"@\w+", " ", text)

  # # 4. Hilangkan simbol hashtag tapi pertahankan katanya
  text = re.sub(r"#", "", text)

  # 5. Hapus angka
  # text = re.sub(r"\d+", " ", text)

  # 6. Hapus emoji
  text = re.sub(
      r"["
      u"\U0001F600-\U0001F64F" # emoticon
      u"\U0001F300-\U0001F5FF" # simbol & pictograph
      u"\U0001F680-\U0001F6FF" # transport
      u"\U0001F1E0-\U0001F1FF" # bendera
      "]+",
      " ",
      text
  )

  # 7. Hapus tanda baca & karakter khusus
  text = re.sub(r"[^a-z0-9\s]", " ", text)

  # 8. Hapus karakter non-ASCII (opsional)
  # text = text.encode("ascii", "ignore").decode("ascii")

  # 9. Normalisasi spasi
  text = re.sub(r"\s+", " ", text).strip()

  return text

ytb["hasil_clean"] = ytb["hasil_casefolding"].apply(clean_text)
ytb.head()
# ytb.to_csv("hasil_clean4.csv", index=False)

,comment,hasil_casefolding,hasil_clean
0,Asap asap untuk test Fan apa namanya gan ?,asap asap untuk test fan apa namanya gan ?,asap asap untuk test fan apa namanya gan
1,5060 bisa sangat unggul price to performanceny...,5060 bisa sangat unggul price to performanceny...,5060 bisa sangat unggul price to performanceny...
2,"Masih pake RX 550 4GB, copotan built-up pula 😂.","masih pake rx 550 4gb, copotan built-up pula 😂.",masih pake rx 550 4gb copotan built up pula
3,"kesimpulan yang gua dapat, RTX lebih unggul, b...","kesimpulan yang gua dapat, rtx lebih unggul, b...",kesimpulan yang gua dapat rtx lebih unggul bay...
4,1070 ti tpi skrng uda artefak 😭,1070 ti tpi skrng uda artefak 😭,1070 ti tpi skrng uda artefak


In [5]:
# TOKENIZATION

ytb["hasil_token"] = ytb["hasil_clean"].apply(lambda x: x.split())
# ytb.head()

In [6]:
# NORMALIZATION

# Load kamus slang dari file csv
slang_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/slangindo/slang_indonesia_v3.csv")
slang_dict = dict(zip(slang_df["slang"], slang_df["formal"]))

def normalized_text(tokens):
  return [slang_dict[word] if word in slang_dict else word for word in tokens]

ytb["hasil_normal"] = ytb["hasil_token"].apply(normalized_text)
# ytb.head(50)
ytb.to_csv("hasil_normal3.csv", index=False)

In [7]:
!pip install Sastrawi

In [8]:
# STOPWORD REMOVAL

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

In [9]:
negation_words = {
    "tidak", "bukan", "jangan", "belum", "kurang", "tanpan"
}

stopwords = stopwords - negation_words

In [10]:
noise_words = {
    "nya", "sih", "dong", "deh", "kok", "nih", "loh", "ya", "kan", "aja", "pun", "lah"
}

stopwords = stopwords.union(noise_words)

In [11]:
def stopword_removal(tokens):
    filtered = []
    for word in tokens:
        if word not in stopwords and len(word) > 2:
            filtered.append(word)
    return filtered

In [12]:
ytb["hasil_stopwords"] = ytb["hasil_normal"].apply(stopword_removal)
# ytb.head()

In [13]:
# STEMMING

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

In [14]:
# Function / Fungsi Stemmer per token
def stemming_token(tokens):
  return [stemmer.stem(word) for word in tokens]

# stemming_commyt = stopwords_commyt.apply(stemming_token)
# stemming_commyt.head(50)

ytb["hasil_stemming"] = ytb["hasil_stopwords"].apply(stemming_token)
# ytb.head()
# ytb.to_csv("data_stemming1.csv", index=False)

In [ ]:
!pip uninstall nlp-id

Found existing installation: nlp-id 0.1.21.0
Uninstalling nlp-id-0.1.21.0:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/nlp_id-0.1.21.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/nlp_id/*
Proceed (Y/n)? y
  Successfully uninstalled nlp-id-0.1.21.0


In [ ]:
# IMPORT LIBRARY

# from nlp_id.lexicon import InsetLexicon
# from nlp_id.lexicon import InsetLexicon

In [15]:
positive_df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/InSet/positive.xlsx")
negative_df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/AnalisisSentimen/AnalisisSentimen_3/Dataset/InSet/negative.xlsx")

print("Jumlah kata positif:", len(positive_df))
print("Jumlah kata negativ:", len(negative_df))

Jumlah kata positif: 3610
Jumlah kata negativ: 6610


In [16]:
# Convert ke dictionary
positive_dict = dict(zip(positive_df['word'], positive_df['weight']))
negative_dict = dict(zip(negative_df['word'], negative_df['weight']))

In [17]:
# Fungsi menghitung skor sentimen

def sentiment_lexicon(tokens):
    score = 0

    for word in tokens:

        # Jika kata ada di lexicon positif
        if word in positive_dict:
            score += positive_dict[word]

        # Jika kata ada di lexicon negatif
        elif word in negative_dict:
            # score -= negative_dict[word]
            score += negative_dict[word]

    # Penentuan label
    if score > 0:
        label = "positif"
    elif score < 0:
        label = "negatif"
    else:
        label = "netral"

    return score, label

In [18]:
hasil_sentimen = ytb["hasil_stemming"].apply(sentiment_lexicon)

# Pisahkan skor dan label
ytb["score_sentiment"] = hasil_sentimen.apply(lambda x: x[0])
ytb["label_sentiment"] = hasil_sentimen.apply(lambda x: x[1])

In [19]:
# MENGGABUNGKAN / JOIN TOKEN KEMBALI

ytb["text_final"] = ytb["hasil_stemming"].apply(lambda x: " ".join(x))
# ytb.head()

In [20]:
ytb[[
    "comment",
    "text_final",
    "score_sentiment",
    "label_sentiment"
]].head(25)
# ytb.to_csv("data_lex2.csv", index=False)

,comment,text_final,score_sentiment,label_sentiment
0,Asap asap untuk test Fan apa namanya gan ?,asap asap test fan apa nama gan,-3,negatif
1,5060 bisa sangat unggul price to performanceny...,5060 sangat unggul price performancenya vramny...,0,netral
2,"Masih pake RX 550 4GB, copotan built-up pula 😂.",pakai 550 4gb copot built,1,positif
3,"kesimpulan yang gua dapat, RTX lebih unggul, b...",simpul gua rtx lebih unggul bayang 8gb non ham...,16,positif
4,1070 ti tpi skrng uda artefak 😭,1070 sekarang artefak,0,netral
5,ngomongnya kecepetan mas jadi kurang jelas 🙏,omong kecepetan mas jadi kurang jelas,-5,negatif
6,Gtx 1070 8GB 2017 ( itu pun kadang VRAMnya nye...,gtx 1070 8gb 2017 kadang vramnya nyentu 8048 l...,8,positif
7,dari 13:22 ke 13:38 di kolom RTX 3060 kok ter...,kolom rtx 3060 deteksi rtx 2080,0,netral
8,nvidia anjai memang ngirit banget 😂\nkalau buk...,nvidia anja memang ngirit banget kalau bukan m...,5,positif
9,RX 6600 the new cesplenk🙌,6600 the new cesplenk,0,netral


In [21]:
# TF IDF

from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)
)

# X = tfidf_vectorizer.fit_transform(yt["label_gpu"])
# yt["sentimen"] = ["positif", "negatif", "netral"]

X = tfidf_vectorizer.fit_transform(ytb["text_final"])

# LABEL
y = ytb["label_sentiment"]

tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
    )
# tfidf_df.head(50)
# print(tfidf_df)

print(tfidf_df.shape)

tfidf_df.head()

(279, 839)


,100,1050,1050ti,1060,1060 6gb,1070,1080p,1080ti,12gb,1440p,...,wkwk,wkwkw,wkwkwk,wkwkwkw,worth,worth enggak,wukong,wuwa,you,youtube
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.286795,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.825367,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
# SPLIT DATA

from sklearn.model_selection import train_test_split

# X = tfidf_df
# y = ytb["text_final"]

X_train, X_test, y_train, y_test = train_test_split(
    # tfidf_df,
    # yt["label_gpu"],
    X, y,
    # yt["text_final"],
    test_size=0.2,
    # test_size=0.8,
    random_state=42
)

In [28]:
# MODEL NAIVE BAYES

from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

In [29]:
# TRAINING

model.fit(X_train, y_train)

MultinomialNB()

In [30]:
# PREDIKSI

y_pred = model.predict(X_test)

In [37]:
# EVALUASI

from sklearn.metrics import accuracy_score, classification_report

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.6607142857142857
              precision    recall  f1-score   support

     negatif       0.71      0.29      0.42        17
      netral       0.00      0.00      0.00         6
     positif       0.67      0.97      0.79        33

    accuracy                           0.66        56
   macro avg       0.46      0.42      0.40        56
weighted avg       0.61      0.66      0.59        56



**Model SVM**

In [38]:
from sklearn.svm import SVC

In [39]:
# Training Model SVM

model_svm = SVC(kernel="linear")

model_svm.fit(X_train, y_train)

SVC(kernel='linear')

In [40]:
# Prediksi

y_pred = model_svm.predict(X_test)

In [41]:
# Evaluasi Model

accuracy = accuracy_score(y_test, y_pred)
print("Akurasi:", accuracy)

Akurasi: 0.6607142857142857


In [42]:
# Classivication Report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     negatif       0.71      0.29      0.42        17
      netral       0.00      0.00      0.00         6
     positif       0.67      0.97      0.79        33

    accuracy                           0.66        56
   macro avg       0.46      0.42      0.40        56
weighted avg       0.61      0.66      0.59        56

